In [ ]:
# Cell 1: Setup
!pip install -q torch transformers datasets sentencepiece pyyaml seqeval conllu accelerate huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Mounted at /content/drive
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, VRAM: 102.0 GB


In [ ]:
# Cell 2: Clone repo + download data
import os, subprocess
if not os.path.exists("/content/kniv-nlp-models"):
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/rustic-ai/kniv-nlp-models.git"], cwd="/content")
from huggingface_hub import snapshot_download
snapshot_download("dragonscale-ai/kniv-corpus-en", repo_type="dataset", allow_patterns="prepared/kniv-deberta-cascade/*", local_dir="/content/kniv-data")
os.chdir("/content/kniv-nlp-models")
os.makedirs("data/prepared", exist_ok=True)
if not os.path.exists("data/prepared/kniv-deberta-cascade"):
    os.symlink("/content/kniv-data/prepared/kniv-deberta-cascade", "data/prepared/kniv-deberta-cascade")
print(f"Ready: {os.getcwd()}")

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Ready: /content/kniv-nlp-models


In [ ]:
# Cell 3: Load leader model (skip DEP — using biaffine now)
import json, torch, sys, os, importlib, shutil
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
import train_srl_encoder
importlib.reload(train_srl_encoder)
from train_srl_encoder import SRLEncoderModel, SRL_TAGS, viterbi_decode
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda")
DRIVE_OUT = "/content/drive/MyDrive/kniv-models/srl_encoder"
LEADER = f"{DRIVE_OUT}/leader_v1"

with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f: vocabs = json.load(f)
pos_labels, ner_labels, dep_labels, cls_labels = vocabs["pos_labels"], vocabs["ner_labels"], vocabs["dep_labels"], vocabs["cls_labels"]

tokenizer = AutoTokenizer.from_pretrained(LEADER)
model = SRLEncoderModel("microsoft/deberta-v3-large", len(SRL_TAGS)).float()
state = torch.load(f"{LEADER}/model.pt", weights_only=True)
model.deberta.load_state_dict(state["deberta"])
model.pred_embedding.load_state_dict(state["pred_embedding"])
model.classifier.load_state_dict(state["classifier"])
H = model.deberta.config.hidden_size
num_layers = model.deberta.config.num_hidden_layers + 1

class ScalarMix(nn.Module):
    def __init__(self, n, favor_range=None):
        super().__init__()
        init = torch.zeros(n)
        if favor_range:
            for i in range(n): init[i] = 2.0 if favor_range[0] <= i <= favor_range[1] else -2.0
        self.weights = nn.Parameter(init)
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, layers):
        w = torch.softmax(self.weights, dim=0)
        return self.scale * sum(wi * li for wi, li in zip(w, layers))

# POS + NER (frozen)
model.pos_scalar_mix = ScalarMix(num_layers); model.pos_head = nn.Linear(H, len(pos_labels))
model.pos_scalar_mix.load_state_dict(state["pos_scalar_mix"]); model.pos_head.load_state_dict(state["pos_head"])
model.ner_scalar_mix = ScalarMix(num_layers)
model.ner_head = nn.Sequential(nn.LayerNorm(H+len(pos_labels)), nn.Linear(H+len(pos_labels), H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, len(ner_labels)))
model.ner_scalar_mix.load_state_dict(state["ner_scalar_mix"]); model.ner_head.load_state_dict(state["ner_head"])

# SRL BiLSTM
model.srl_lstm = nn.LSTM(H, 256, num_layers=1, bidirectional=True, batch_first=True)
model.srl_proj = nn.Linear(512, H)
model.srl_lstm.load_state_dict(state["srl_lstm"]); model.srl_proj.load_state_dict(state["srl_proj"])

model = model.to(device)
print(f"Leader loaded: POS=0.966, NER=0.846, SRL=0.802, DEP=0.922 (biaffine, separate)")

# Load data for later cells
with open("data/prepared/kniv-deberta-cascade/ud_train.json") as f: ud_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ud_dev.json") as f: ud_dev = json.load(f)
collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}

head_scores = {"pos": {"score": 0.966, "metric": "accuracy"}, "ner": {"score": 0.846, "metric": "F1"},
               "srl": {"score": 0.802, "metric": "F1"}, "dep": {"score": 0.922, "metric": "UAS"},
               "cls": {"score": 0.0, "metric": "macro_f1"}}
print("Ready for CLS training.")

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Leader loaded: POS=0.966, NER=0.846, SRL=0.802, DEP=0.922 (biaffine, separate)
Ready for CLS training.


In [ ]:
# Cell 4: DEP v2 — 200K silver + 12.5K gold, MLP-2, 10 epochs
# Download silver DEP data
from huggingface_hub import hf_hub_download
silver_dep_path = "data/prepared/kniv-deberta-cascade/dep_silver_spacy.json"
if not os.path.exists(silver_dep_path):
    hf_hub_download("dragonscale-ai/kniv-corpus-en", "prepared/kniv-deberta-cascade/dep_silver_spacy.json",
                    repo_type="dataset", local_dir="/content/kniv-data")
    if not os.path.exists(silver_dep_path):
        os.symlink("/content/kniv-data/prepared/kniv-deberta-cascade/dep_silver_spacy.json", silver_dep_path)

# Load silver + gold DEP data
import random
with open(silver_dep_path) as f: dep_silver = json.load(f)
random.seed(42)
dep_silver_sub = random.sample(dep_silver, min(100000, len(dep_silver)))
# Combine with gold (upsampled 3x for emphasis)
dep_combined = dep_silver_sub + ud_train * 3
random.shuffle(dep_combined)
print(f"DEP combined: {len(dep_combined):,} (100K silver + {len(ud_train)*3:,} gold x3)")

# New DEP head: MLP-2 (more capacity for 1440 classes)
model.dep_scalar_mix = ScalarMix(num_layers, favor_range=(12, 18))
in_dim = H + len(pos_labels) + len(ner_labels)
model.dep_head = nn.Sequential(
    nn.LayerNorm(in_dim),
    nn.Linear(in_dim, H), nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H // 2, len(dep_labels)),
)
model = model.to(device)

for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "dep_scalar_mix" in n or "dep_head" in n: p.requires_grad = True
print(f"DEP MLP-2 trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

dep_combined_loader = DataLoader(TokenDataset(dep_combined, tokenizer, dep_map, "dep_labels"), batch_size=64, shuffle=True, collate_fn=collate)

dep_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(dep_params, lr=1e-3)
epochs = 10
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(dep_combined_loader)*0.1), epochs*len(dep_combined_loader))

head_scores["dep"] = {"score": 0.770, "metric": "accuracy"}
print(f"\nDEP v2: {epochs} epochs, {len(dep_combined_loader)} batches/epoch, MLP-2, LR=1e-3\n")

best_dep = 0.770
for epoch in range(epochs):
    model.train(); tl = 0.0
    for batch in tqdm(dep_combined_loader, desc=f"DEP E{epoch+1}"):
        logits = fwd_dep(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(dep_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step(); tl += loss.item()
    model.eval(); correct, total = 0, 0
    with torch.no_grad():
        for batch in dep_dev_loader:
            logits = fwd_dep(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu(); mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item(); total += mask.sum().item()
    acc = correct / total
    print(f"DEP E{epoch+1}/{epochs} — Loss: {tl/len(dep_combined_loader):.4f}, Acc: {acc:.4f}")
    if acc > best_dep:
        best_dep = acc; state["dep_scalar_mix"] = model.dep_scalar_mix.state_dict(); state["dep_head"] = model.dep_head.state_dict()
        head_scores["dep"] = {"score": round(best_dep, 4), "metric": "accuracy"}
        save_leader(state, f"v2.3_dep{round(best_dep*1000)}", head_scores)
print(f"\nBest DEP Acc: {best_dep:.4f}")

prepared/kniv-deberta-cascade/dep_silver(…):   0%|          | 0.00/182M [00:00<?, ?B/s]

DEP combined: 137,632 (100K silver + 37,632 gold x3)
DEP MLP-2 trainable: 2,370,598

DEP v2: 10 epochs, 2151 batches/epoch, MLP-2, LR=1e-3



DEP E1: 100%|██████████| 2151/2151 [06:16<00:00,  5.71it/s]


DEP E1/10 — Loss: 1.6883, Acc: 0.6382


DEP E2: 100%|██████████| 2151/2151 [06:16<00:00,  5.71it/s]


...

DEP E5/10 — Loss: 0.6301, Acc: 0.7191


DEP E6: 100%|██████████| 2151/2151 [06:16<00:00,  5.71it/s]


DEP E6/10 — Loss: 0.5940, Acc: 0.7125


DEP E7:   9%|▉         | 202/2151 [00:35<05:43,  5.68it/s]


KeyboardInterrupt: 

In [ ]:
# Cell 5: DEP v3 — MLP-2 on gold-only, 30 epochs
# Fresh MLP-2 head, gold UD EWT only (12.5K), long training

# New DEP head: MLP-2
model.dep_scalar_mix = ScalarMix(num_layers, favor_range=(12, 18))
in_dim = H + len(pos_labels) + len(ner_labels)
model.dep_head = nn.Sequential(
    nn.LayerNorm(in_dim),
    nn.Linear(in_dim, H), nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H // 2, len(dep_labels)),
)
model = model.to(device)

for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "dep_scalar_mix" in n or "dep_head" in n: p.requires_grad = True
print(f"DEP MLP-2 trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Gold only
dep_gold_loader = DataLoader(TokenDataset(ud_train, tokenizer, dep_map, "dep_labels"), batch_size=64, shuffle=True, collate_fn=collate)

dep_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(dep_params, lr=1e-3)
epochs = 30
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(dep_gold_loader)*0.1), epochs*len(dep_gold_loader))

print(f"\nDEP v3 (MLP-2, gold only): {epochs} epochs, {len(dep_gold_loader)} batches/epoch\n")

best_dep = 0.0
for epoch in range(epochs):
    model.train(); tl = 0.0
    for batch in tqdm(dep_gold_loader, desc=f"DEP E{epoch+1}"):
        logits = fwd_dep(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(dep_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step(); tl += loss.item()
    model.eval(); correct, total = 0, 0
    with torch.no_grad():
        for batch in dep_dev_loader:
            logits = fwd_dep(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            preds = logits.argmax(dim=-1).cpu(); mask = batch["labels"] != -100
            correct += (preds[mask] == batch["labels"][mask]).sum().item(); total += mask.sum().item()
    acc = correct / total
    if (epoch + 1) % 5 == 0 or acc > best_dep:
        print(f"DEP E{epoch+1}/{epochs} — Loss: {tl/len(dep_gold_loader):.4f}, Acc: {acc:.4f}")
    if acc > best_dep:
        best_dep = acc; state["dep_scalar_mix"] = model.dep_scalar_mix.state_dict(); state["dep_head"] = model.dep_head.state_dict()
        head_scores["dep"] = {"score": round(best_dep, 4), "metric": "accuracy"}
        save_leader(state, f"v2.3_dep{round(best_dep*1000)}", head_scores)

print(f"\nBest DEP Acc: {best_dep:.4f}")

In [ ]:
# Cell 6: DEP v4 — Biaffine dependency parser (fixed .reshape)

deprel_list = ["root", "acl", "acl:relcl", "advcl", "advcl:relcl", "advmod", "amod", "appos",
    "aux", "aux:pass", "case", "cc", "cc:preconj", "ccomp", "compound", "compound:prt",
    "conj", "cop", "csubj", "csubj:outer", "csubj:pass", "dep", "det", "det:predet",
    "discourse", "dislocated", "expl", "fixed", "flat", "goeswith", "iobj", "list", "mark",
    "nmod", "nmod:desc", "nmod:npmod", "nmod:poss", "nmod:tmod", "nsubj", "nsubj:outer",
    "nsubj:pass", "nummod", "obj", "obl", "obl:agent", "obl:npmod", "obl:tmod",
    "orphan", "parataxis", "punct", "reparandum", "vocative", "xcomp"]
deprel_map = {r: i for i, r in enumerate(deprel_list)}
num_deprels = len(deprel_list)

class Biaffine(nn.Module):
    def __init__(self, in_dim, out_dim=1):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(out_dim, in_dim + 1, in_dim + 1))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, h_dep, h_head):
        B, S, D = h_dep.size()
        h_dep = torch.cat([h_dep, torch.ones(B, S, 1, device=h_dep.device)], dim=-1)
        h_head = torch.cat([h_head, torch.ones(B, S, 1, device=h_head.device)], dim=-1)
        scores = torch.einsum("bxi,oij,byj->boxy", h_dep, self.weight, h_head)
        if scores.size(1) == 1:
            return scores.squeeze(1).contiguous()  # [B, S, S]
        else:
            return scores.permute(0, 2, 3, 1).contiguous()  # [B, S, S, out]

class BiaffineDEPHead(nn.Module):
    def __init__(self, hidden_size, arc_dim=512, label_dim=128, num_labels=53, dropout=0.1):
        super().__init__()
        self.arc_dep = nn.Sequential(nn.Linear(hidden_size, arc_dim), nn.LayerNorm(arc_dim), nn.GELU(), nn.Dropout(dropout))
        self.arc_head = nn.Sequential(nn.Linear(hidden_size, arc_dim), nn.LayerNorm(arc_dim), nn.GELU(), nn.Dropout(dropout))
        self.label_dep = nn.Sequential(nn.Linear(hidden_size, label_dim), nn.LayerNorm(label_dim), nn.GELU(), nn.Dropout(dropout))
        self.label_head = nn.Sequential(nn.Linear(hidden_size, label_dim), nn.LayerNorm(label_dim), nn.GELU(), nn.Dropout(dropout))
        self.biaffine_arc = Biaffine(arc_dim, out_dim=1)
        self.biaffine_label = Biaffine(label_dim, out_dim=num_labels)
    def forward(self, hidden):
        return self.biaffine_arc(self.arc_dep(hidden), self.arc_head(hidden)), \
               self.biaffine_label(self.label_dep(hidden), self.label_head(hidden))

in_dim = H + len(pos_labels) + len(ner_labels)
model.dep_scalar_mix = ScalarMix(num_layers, favor_range=(12, 18))
model.dep_proj = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, H), nn.GELU())
model.dep_biaffine = BiaffineDEPHead(H, arc_dim=512, label_dim=128, num_labels=num_deprels)
model = model.to(device)

for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "dep_scalar_mix" in n or "dep_proj" in n or "dep_biaffine" in n: p.requires_grad = True
print(f"Biaffine DEP trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def fwd_dep_biaffine(input_ids, attention_mask):
    out = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    dep_h = model.dep_scalar_mix(out.hidden_states)
    pos_h = model.pos_scalar_mix(out.hidden_states)
    pos_p = torch.softmax(model.pos_head(pos_h), dim=-1).detach()
    ner_h = model.ner_scalar_mix(out.hidden_states)
    ner_p = torch.softmax(model.ner_head(torch.cat([ner_h, pos_p], dim=-1)), dim=-1).detach()
    return model.dep_biaffine(model.dep_proj(torch.cat([dep_h, pos_p, ner_p], dim=-1)))

class BiaffineDEPDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, deprel_map, max_length=128):
        self.examples, self.tokenizer, self.deprel_map, self.max_length = examples, tokenizer, deprel_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        words, heads, deprels = ex["words"], ex["heads"], ex["deprels"]
        enc = self.tokenizer(words, is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        word_ids = enc.word_ids()
        seq_len = enc["input_ids"].size(1)
        head_labels = torch.full((seq_len,), -1, dtype=torch.long)
        rel_labels = torch.full((seq_len,), -1, dtype=torch.long)
        word_to_token = {}
        prev = None
        for k, wid in enumerate(word_ids):
            if wid is not None and wid != prev: word_to_token[wid] = k
            prev = wid
        prev = None
        for k, wid in enumerate(word_ids):
            if wid is not None and wid != prev and wid < len(heads):
                hw = heads[wid]
                if hw == -1: head_labels[k] = k
                elif hw in word_to_token: head_labels[k] = word_to_token[hw]
                rel_labels[k] = self.deprel_map.get(deprels[wid] if wid < len(deprels) else "dep", 0)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0),
                "head_labels": head_labels, "rel_labels": rel_labels}

dep_collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
dep_train_biaff = torch.utils.data.DataLoader(BiaffineDEPDataset(ud_train, tokenizer, deprel_map), batch_size=32, shuffle=True, collate_fn=dep_collate)
dep_dev_biaff = torch.utils.data.DataLoader(BiaffineDEPDataset(ud_dev, tokenizer, deprel_map), batch_size=32, collate_fn=dep_collate)

dep_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(dep_params, lr=1e-3)
epochs = 20
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(dep_train_biaff)*0.1), epochs*len(dep_train_biaff))
arc_loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
rel_loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
print(f"\nBiaffine DEP: {epochs} epochs, {len(dep_train_biaff)} batches/epoch\n")

best_uas = 0.0
for epoch in range(epochs):
    model.train(); tl = 0.0
    for batch in tqdm(dep_train_biaff, desc=f"DEP E{epoch+1}"):
        arc_scores, label_scores = fwd_dep_biaffine(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        head_labels = batch["head_labels"].to(device)
        rel_labels = batch["rel_labels"].to(device)
        B, S = head_labels.size()
        arc_loss = arc_loss_fn(arc_scores.reshape(B * S, S), head_labels.reshape(B * S))
        mask = head_labels >= 0
        if mask.any():
            gold_heads = head_labels.clamp(min=0).unsqueeze(-1).unsqueeze(-1).expand(B, S, 1, num_deprels)
            token_label_scores = label_scores.gather(2, gold_heads).squeeze(2)
            rel_loss = rel_loss_fn(token_label_scores.reshape(B * S, num_deprels), rel_labels.reshape(B * S))
        else:
            rel_loss = torch.tensor(0.0, device=device)
        loss = arc_loss + rel_loss
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); tl += loss.item()

    model.eval(); arc_correct, rel_correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in dep_dev_biaff:
            arc_scores, label_scores = fwd_dep_biaffine(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            head_labels, rel_labels = batch["head_labels"], batch["rel_labels"]
            pred_heads = arc_scores.argmax(dim=-1).cpu()
            mask = head_labels >= 0
            arc_correct += (pred_heads[mask] == head_labels[mask]).sum().item()
            B, S = head_labels.size()
            ph = pred_heads.clamp(min=0).unsqueeze(-1).unsqueeze(-1).expand(B, S, 1, num_deprels)
            pred_rels = label_scores.cpu().gather(2, ph).squeeze(2).argmax(dim=-1)
            both = (pred_heads[mask] == head_labels[mask]) & (pred_rels[mask] == rel_labels[mask])
            rel_correct += both.sum().item(); total += mask.sum().item()
    uas = arc_correct / total; las = rel_correct / total
    if (epoch + 1) % 5 == 0 or uas > best_uas:
        print(f"DEP E{epoch+1}/{epochs} — Loss: {tl/len(dep_train_biaff):.4f}, UAS: {uas:.4f}, LAS: {las:.4f}")
    if uas > best_uas:
        best_uas = uas
        state["dep_scalar_mix"] = model.dep_scalar_mix.state_dict()
        state["dep_proj"] = model.dep_proj.state_dict()
        state["dep_biaffine"] = model.dep_biaffine.state_dict()
        head_scores["dep"] = {"score": round(best_uas, 4), "metric": "UAS", "las": round(las, 4)}
        save_leader(state, f"v3.0_dep_uas{round(best_uas*1000)}", head_scores)
print(f"\nBest UAS: {best_uas:.4f}")

Biaffine DEP trainable: 3,566,780

Biaffine DEP: 20 epochs, 392 batches/epoch



DEP E1: 100%|██████████| 392/392 [00:36<00:00, 10.72it/s]


DEP E1/20 — Loss: 1.6792, UAS: 0.8604, LAS: 0.8154
  Leader saved: v3.0_dep_uas860


DEP E2: 100%|██████████| 392/392 [00:36<00:00, 10.74it/s]


DEP E2/20 — Loss: 0.7430, UAS: 0.8782, LAS: 0.8368
  Leader saved: v3.0_dep_uas878


...

DEP E17/20 — Loss: 0.0513, UAS: 0.9202, LAS: 0.8971
  Leader saved: v3.0_dep_uas920


DEP E19: 100%|██████████| 392/392 [00:36<00:00, 10.77it/s]


DEP E19/20 — Loss: 0.0396, UAS: 0.9217, LAS: 0.8990
  Leader saved: v3.0_dep_uas922


DEP E20: 100%|██████████| 392/392 [00:36<00:00, 10.75it/s]


DEP E20/20 — Loss: 0.0354, UAS: 0.9217, LAS: 0.8992

Best UAS: 0.9217


In [ ]:
# Cell 7: CLS A/B — new 8-label scheme, SGD+GPT data, 3 pooling methods
from huggingface_hub import hf_hub_download
import random

cls_path = "data/prepared/kniv-deberta-cascade/cls_sgd_mwoz_train.json"
if not os.path.exists(cls_path):
    hf_hub_download("dragonscale-ai/kniv-corpus-en", "prepared/kniv-deberta-cascade/cls_sgd_mwoz_train.json",
                    repo_type="dataset", local_dir="/content/kniv-data")
    if not os.path.exists(cls_path):
        os.symlink("/content/kniv-data/prepared/kniv-deberta-cascade/cls_sgd_mwoz_train.json", cls_path)

with open(cls_path) as f: cls_data = json.load(f)
random.seed(42); random.shuffle(cls_data)
cls_dev_data = cls_data[:4000]
cls_train_data = cls_data[4000:]
new_cls_labels = ["inform", "request", "question", "confirm", "reject", "offer", "social", "status"]
cls_map = {l: i for i, l in enumerate(new_cls_labels)}
print(f"CLS: train={len(cls_train_data):,}, dev={len(cls_dev_data):,}, labels={len(new_cls_labels)}")

from sklearn.metrics import f1_score as clf_f1, classification_report

class CLSDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        text = ex.get("text", "")
        prev_text = ex.get("prev_text")
        if prev_text:
            enc = self.tokenizer(prev_text, text, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        else:
            enc = self.tokenizer(text, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.label_map[ex["cls_label"]], dtype=torch.long)}

cls_collate = lambda feats: {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
cls_train_loader = torch.utils.data.DataLoader(CLSDataset(cls_train_data, tokenizer, cls_map), batch_size=64, shuffle=True, collate_fn=cls_collate)
cls_dev_loader = torch.utils.data.DataLoader(CLSDataset(cls_dev_data, tokenizer, cls_map), batch_size=64, collate_fn=cls_collate)

class AttentionPool(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)
    def forward(self, hidden, mask):
        scores = self.attn(hidden).squeeze(-1)
        scores = scores.masked_fill(~mask.bool(), -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(dim=1)

def run_cls_experiment(name, pool_type, epochs=5):
    print(f"\n{'='*50}\nCLS: {name}\n{'='*50}")
    model.cls_scalar_mix = ScalarMix(num_layers)
    if pool_type == "attention":
        model.cls_pool = AttentionPool(H)
    model.cls_head = nn.Sequential(nn.LayerNorm(H), nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.1), nn.Linear(H // 2, len(new_cls_labels)))
    model.to(device)
    for p in model.parameters(): p.requires_grad = False
    for n, p in model.named_parameters():
        if "cls_scalar_mix" in n or "cls_head" in n or "cls_pool" in n: p.requires_grad = True
    print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    def fwd(input_ids, attention_mask):
        out = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        h = model.cls_scalar_mix(out.hidden_states)
        if pool_type == "cls_token": return model.cls_head(h[:, 0, :])
        elif pool_type == "attention": return model.cls_head(model.cls_pool(h, attention_mask))
        else:
            mask = attention_mask.unsqueeze(-1).float()
            return model.cls_head((h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1))
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    sched = get_linear_schedule_with_warmup(opt, int(epochs*len(cls_train_loader)*0.1), epochs*len(cls_train_loader))
    loss_fn = nn.CrossEntropyLoss()
    best_f1 = 0.0
    for epoch in range(epochs):
        model.train(); tl = 0.0
        for batch in tqdm(cls_train_loader, desc=f"E{epoch+1}", leave=False):
            logits = fwd(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            loss = loss_fn(logits, batch["labels"].to(device))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tl += loss.item()
        model.eval(); all_g, all_p = [], []
        with torch.no_grad():
            for batch in cls_dev_loader:
                logits = fwd(batch["input_ids"].to(device), batch["attention_mask"].to(device))
                all_p.extend(logits.argmax(dim=-1).cpu().tolist()); all_g.extend(batch["labels"].tolist())
        mf1 = clf_f1(all_g, all_p, average="macro")
        print(f"  E{epoch+1}/{epochs} — Loss: {tl/len(cls_train_loader):.4f}, Macro F1: {mf1:.4f}")
        if mf1 > best_f1: best_f1 = mf1
    print(classification_report(all_g, all_p, target_names=new_cls_labels, digits=3))
    print(f"  Best: {best_f1:.4f}")
    if hasattr(model, "cls_pool"): del model.cls_pool
    del model.cls_scalar_mix, model.cls_head; torch.cuda.empty_cache()
    return best_f1

results = {}
results["Mean pool"] = run_cls_experiment("Mean pool", "mean")
results["[CLS] token"] = run_cls_experiment("[CLS] token", "cls_token")
results["Attention pool"] = run_cls_experiment("Attention pool", "attention")

print(f"\n{'='*50}\nCLS RESULTS (8 labels, SGD+GPT data)\n{'='*50}")
for name, f1 in sorted(results.items(), key=lambda x: -x[1]):
    bar = "█" * int(f1 * 50)
    print(f"  {f1:.4f}  {bar}  {name}")

CLS: train=60,000, dev=4,000, labels=8

CLS: Mean pool
Trainable: 530,978


  E1/5 — Loss: 0.8171, Macro F1: 0.8421


  E2/5 — Loss: 0.4160, Macro F1: 0.8735


  E3/5 — Loss: 0.3197, Macro F1: 0.8935


  E4/5 — Loss: 0.2646, Macro F1: 0.9099


...

  E1/5 — Loss: 0.7121, Macro F1: 0.8680


  E2/5 — Loss: 0.3125, Macro F1: 0.9043


  E3/5 — Loss: 0.2417, Macro F1: 0.9235


  E4/5 — Loss: 0.1949, Macro F1: 0.9279


  E5/5 — Loss: 0.1608, Macro F1: 0.9307
              precision    recall  f1-score   support

      inform      0.856     0.885     0.870       538
     request      0.924     0.915     0.919       492
    question      0.845     0.830     0.837       487
     confirm      0.959     0.952     0.956       520
      reject      0.973     0.984     0.978       485
       offer      0.973     0.981     0.977       484
      social      0.930     0.925     0.928       506
      status      0.985     0.973     0.979       488

    accuracy                          0.930      4000
   macro avg      0.931     0.931     0.931      4000
weighted avg      0.930     0.930     0.930      4000

  Best: 0.9307

CLS RESULTS (8 labels, SGD+GPT data)
  0.9307  ██████████████████████████████████████████████  Attention pool
  0.9141  █████████████████████████████████████████████  Mean pool
  0.8494  ██████████████████████████████████████████  [CLS] token


In [ ]:
# Cell 8: Save CLS (attention pool) to leader
import shutil

# Retrain best config (attention pool) and save
model.cls_scalar_mix = ScalarMix(num_layers)
model.cls_pool = AttentionPool(H)
model.cls_head = nn.Sequential(nn.LayerNorm(H), nn.Linear(H, H // 2), nn.GELU(), nn.Dropout(0.1), nn.Linear(H // 2, len(new_cls_labels)))
model.to(device)

for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "cls_scalar_mix" in n or "cls_head" in n or "cls_pool" in n: p.requires_grad = True

def fwd_cls(input_ids, attention_mask):
    out = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    h = model.cls_scalar_mix(out.hidden_states)
    return model.cls_head(model.cls_pool(h, attention_mask))

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
epochs = 5
sched = get_linear_schedule_with_warmup(opt, int(epochs*len(cls_train_loader)*0.1), epochs*len(cls_train_loader))
loss_fn = nn.CrossEntropyLoss()

print("Training CLS (attention pool) for leader save...\n")
best_f1 = 0.0
for epoch in range(epochs):
    model.train(); tl = 0.0
    for batch in tqdm(cls_train_loader, desc=f"E{epoch+1}", leave=False):
        logits = fwd_cls(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits, batch["labels"].to(device))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tl += loss.item()
    model.eval(); all_g, all_p = [], []
    with torch.no_grad():
        for batch in cls_dev_loader:
            logits = fwd_cls(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            all_p.extend(logits.argmax(dim=-1).cpu().tolist()); all_g.extend(batch["labels"].tolist())
    from sklearn.metrics import f1_score as clf_f1
    mf1 = clf_f1(all_g, all_p, average="macro")
    print(f"  E{epoch+1}/{epochs} — Macro F1: {mf1:.4f}")
    if mf1 > best_f1: best_f1 = mf1

# Save to leader
state["cls_scalar_mix"] = model.cls_scalar_mix.state_dict()
state["cls_pool"] = model.cls_pool.state_dict()
state["cls_head"] = model.cls_head.state_dict()
head_scores["cls"] = {"score": round(best_f1, 4), "metric": "macro_f1", "labels": new_cls_labels}

def save_leader(sd, version, scores):
    torch.save(sd, f"{LEADER}/model.pt")
    meta = {"heads": scores, "encoder": "DeBERTa-v3-large (SRL-tuned)", "version": version}
    with open(f"{LEADER}/metadata.json", "w") as f: json.dump(meta, f, indent=2)
    bk = f"{DRIVE_OUT}/leader_{version}"
    os.makedirs(bk, exist_ok=True)
    shutil.copy2(f"{LEADER}/model.pt", f"{bk}/model.pt")
    shutil.copy2(f"{LEADER}/metadata.json", f"{bk}/metadata.json")
    print(f"  Leader saved: {version}")

save_leader(state, f"v3.1_cls{round(best_f1*1000)}", head_scores)

print(f"\n{'='*50}")
print("LEADER v3.1")
print(f"{'='*50}")
for h, info in head_scores.items():
    print(f"  {h.upper():>4}: {info['score']:.4f} ({info['metric']})")

Training CLS (attention pool) for leader save...



  E1/5 — Macro F1: 0.8851


  E2/5 — Macro F1: 0.9029


  E3/5 — Macro F1: 0.9225


  E4/5 — Macro F1: 0.9264


  E5/5 — Macro F1: 0.9301
  Leader saved: v3.1_cls930

LEADER v3.1
   POS: 0.9660 (accuracy)
   NER: 0.8460 (F1)
   SRL: 0.8020 (F1)
   DEP: 0.9220 (UAS)
   CLS: 0.9301 (macro_f1)


In [ ]:
# Cell 9: NER + BiLSTM adapter, 20 epochs
# Same pattern as SRL BiLSTM: encoder → BiLSTM → residual → MLP → VD

lstm_hidden = 256
model.ner_lstm = nn.LSTM(H, lstm_hidden, num_layers=1, bidirectional=True, batch_first=True)
model.ner_proj = nn.Linear(lstm_hidden * 2, H)

# Fresh NER head with BiLSTM (scalar mix loaded from leader)
model.ner_scalar_mix = ScalarMix(num_layers)
model.ner_scalar_mix.load_state_dict(state["ner_scalar_mix"])
model.ner_head_lstm = nn.Sequential(nn.LayerNorm(H + len(pos_labels)), nn.Linear(H + len(pos_labels), H), nn.GELU(), nn.Dropout(0.1), nn.Linear(H, len(ner_labels)))
# Init from existing NER head weights
model.ner_head_lstm.load_state_dict(state["ner_head"])
model = model.to(device)

for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "ner_lstm" in n or "ner_proj" in n or "ner_head_lstm" in n or "ner_scalar_mix" in n:
        p.requires_grad = True
print(f"NER+BiLSTM trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def fwd_ner_lstm(input_ids, attention_mask):
    out = model.deberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    ner_h = model.ner_scalar_mix(out.hidden_states)
    # BiLSTM adapter with residual
    lstm_out, _ = model.ner_lstm(ner_h)
    adapted = model.ner_proj(lstm_out) + ner_h
    # POS cascade
    pos_h = model.pos_scalar_mix(out.hidden_states)
    pos_p = torch.softmax(model.pos_head(pos_h), dim=-1).detach()
    return model.ner_head_lstm(torch.cat([adapted, pos_p], dim=-1))

# Data
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_train.json") as f: ner_train = json.load(f)
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_dev.json") as f: ner_dev = json.load(f)

class NERDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex["words"], is_split_into_words=True, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        aligned, prev = [], None
        for wid in enc.word_ids():
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(self.label_map.get(ex["ner_tags"][wid] if wid < len(ex["ner_tags"]) else "O", 0))
            else: aligned.append(-100)
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(aligned, dtype=torch.long)}

ner_map = {l: i for i, l in enumerate(ner_labels)}
ner_train_loader = DataLoader(NERDataset(ner_train, tokenizer, ner_map), batch_size=64, shuffle=True, collate_fn=collate)
ner_dev_loader = DataLoader(NERDataset(ner_dev[:2000], tokenizer, ner_map), batch_size=64, collate_fn=collate)

from seqeval.metrics import f1_score as seq_f1

ner_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(ner_params, lr=1e-3)
epochs = 20
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(ner_train_loader)*0.1), epochs*len(ner_train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nNER+BiLSTM: {epochs} epochs, {len(ner_train_loader)} batches/epoch\n")

best_ner = 0.846
for epoch in range(epochs):
    model.train(); tl = 0.0
    for batch in tqdm(ner_train_loader, desc=f"NER E{epoch+1}"):
        logits = fwd_ner_lstm(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(ner_labels)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step(); tl += loss.item()
    model.eval(); all_g, all_p = [], []
    with torch.no_grad():
        for batch in ner_dev_loader:
            logits = fwd_ner_lstm(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            logits_cpu = logits.cpu(); labs = batch["labels"]
            for j in range(labs.size(0)):
                vi = (labs[j] != -100).nonzero(as_tuple=True)[0]
                if len(vi) == 0: continue
                vl = logits_cpu[j, vi]
                vd = viterbi_decode(vl, ner_map)
                all_g.append([ner_labels[labs[j, k]] for k in vi.tolist()])
                all_p.append([ner_labels[vd[ki]] if vd[ki] < len(ner_labels) else "O" for ki in range(len(vi))])
    f1 = seq_f1(all_g, all_p)
    if (epoch+1) % 5 == 0 or f1 > best_ner:
        print(f"NER E{epoch+1}/{epochs} — Loss: {tl/len(ner_train_loader):.4f}, F1: {f1:.4f}")
    if f1 > best_ner:
        best_ner = f1
        state["ner_lstm"] = model.ner_lstm.state_dict()
        state["ner_proj"] = model.ner_proj.state_dict()
        state["ner_head_lstm"] = model.ner_head_lstm.state_dict()
        state["ner_scalar_mix"] = model.ner_scalar_mix.state_dict()
        head_scores["ner"] = {"score": round(best_ner, 4), "metric": "F1"}
        save_leader(state, f"v3.2_ner{round(best_ner*1000)}", head_scores)
print(f"\nBest NER F1: {best_ner:.4f}")

NER+BiLSTM trainable: 4,257,889

NER+BiLSTM: 20 epochs, 3049 batches/epoch



NER E3: 100%|██████████| 3049/3049 [09:05<00:00,  5.59it/s]


NER E3/20 — Loss: 0.0991, F1: 0.8519
  Leader saved: v3.2_ner852


NER E5: 100%|██████████| 3049/3049 [09:05<00:00,  5.59it/s]


NER E5/20 — Loss: 0.0857, F1: 0.8554
  Leader saved: v3.2_ner855


NER E9: 100%|██████████| 3049/3049 [09:05<00:00,  5.59it/s]


NER E9/20 — Loss: 0.0658, F1: 0.8597
  Leader saved: v3.2_ner860


NER E10: 100%|██████████| 3049/3049 [09:05<00:00,  5.59it/s]


NER E10/20 — Loss: 0.0614, F1: 0.8520


NER E11:  39%|███▉      | 1192/3049 [03:33<05:31,  5.59it/s]

SRL data: 363,488 train (200,000 silver + 163,488 gold x4), 1,236 dev
SRL Two-Segment trainable: 1,094,724

SRL Two-Segment: 10 epochs, 5680 batches/epoch



SRL2 E1: 100%|██████████| 5680/5680 [16:28<00:00,  5.74it/s]


SRL2 E1/10 — Loss: 0.9156, F1: 0.5987


SRL2 E2: 100%|██████████| 5680/5680 [16:28<00:00,  5.75it/s]


SRL2 E2/10 — Loss: 0.7101, F1: 0.6533


...

SRL2 E7/10 — Loss: 0.5967, F1: 0.6679


SRL2 E8: 100%|██████████| 5680/5680 [16:28<00:00,  5.75it/s]


SRL2 E8/10 — Loss: 0.5870, F1: 0.6785


SRL2 E9:  77%|███████▋  | 4401/5680 [12:46<03:42,  5.74it/s]


KeyboardInterrupt: 

In [ ]:
# Cell 11: SRL v2 — Retrain BiLSTM head with more epochs + fresh ScalarMix
# Current baseline: 0.802 F1 (pred_embedding + BiLSTM + 10 epochs)
# Strategy: 20 epochs, fresh trainable ScalarMix, larger BiLSTM (512 hidden)

import random
from seqeval.metrics import f1_score as seq_f1

# Load SRL data
with open("data/prepared/kniv-deberta-cascade/srl_allennlp_silver.json") as f: srl_silver = json.load(f)
with open("data/prepared/kniv-deberta-cascade/srl_train.json") as f: srl_gold = json.load(f)
with open("data/prepared/kniv-deberta-cascade/srl_dev.json") as f: srl_dev = json.load(f)

random.seed(42)
srl_silver_sub = random.sample(srl_silver, min(200000, len(srl_silver)))
srl_combined = srl_silver_sub + srl_gold * 4
random.shuffle(srl_combined)
print(f"SRL data: {len(srl_combined):,} train, {len(srl_dev):,} dev")

srl_tag_list = list(SRL_TAGS)
srl_map = {t: i for i, t in enumerate(srl_tag_list)}

class SRLDataset(Dataset):
    def __init__(self, examples, tokenizer, label_map, max_length=128):
        self.examples, self.tokenizer, self.label_map, self.max_length = examples, tokenizer, label_map, max_length
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        words, tags = ex["words"], ex["srl_tags"]
        pred_idx = ex["predicate_idx"]
        enc = self.tokenizer(words, is_split_into_words=True, max_length=self.max_length,
                             padding="max_length", truncation=True, return_tensors="pt")
        word_ids = enc.word_ids()
        # Build predicate mask (aligned to subwords)
        pred_mask = torch.zeros(self.max_length, dtype=torch.long)
        aligned, prev = [], None
        for k, wid in enumerate(word_ids):
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(self.label_map.get(tags[wid] if wid < len(tags) else "O", 0))
                if wid == pred_idx: pred_mask[k] = 1
            else:
                aligned.append(-100)
                if wid == pred_idx: pred_mask[k] = 1
            prev = wid
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(aligned, dtype=torch.long), "pred_mask": pred_mask}

srl_train_loader = DataLoader(SRLDataset(srl_combined, tokenizer, srl_map), batch_size=64, shuffle=True, collate_fn=collate)
srl_dev_loader = DataLoader(SRLDataset(srl_dev, tokenizer, srl_map), batch_size=64, collate_fn=collate)

# Fresh SRL components — larger BiLSTM (512 hidden vs 256)
lstm_hidden = 512
model.srl_scalar_mix2 = ScalarMix(num_layers, favor_range=(20, 24))
model.srl_lstm2 = nn.LSTM(H, lstm_hidden, num_layers=1, bidirectional=True, batch_first=True)
model.srl_proj2 = nn.Linear(lstm_hidden * 2, H)
model.srl_head2 = nn.Sequential(
    nn.LayerNorm(H), nn.Linear(H, H), nn.GELU(), nn.Dropout(0.1),
    nn.Linear(H, len(srl_tag_list))
)
model = model.to(device)

for p in model.parameters(): p.requires_grad = False
for n, p in model.named_parameters():
    if "srl_scalar_mix2" in n or "srl_lstm2" in n or "srl_proj2" in n or "srl_head2" in n:
        p.requires_grad = True
print(f"SRL v2 trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def fwd_srl_v2(input_ids, attention_mask, pred_mask):
    # Inject predicate embedding (already in encoder from SRL-first training)
    embed = model.deberta.embeddings(input_ids)
    embed = embed + model.pred_embedding(pred_mask)
    out = model.deberta(inputs_embeds=embed, attention_mask=attention_mask, output_hidden_states=True)
    h = model.srl_scalar_mix2(out.hidden_states)
    lstm_out, _ = model.srl_lstm2(h)
    adapted = model.srl_proj2(lstm_out) + h  # residual
    return model.srl_head2(adapted)

srl_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(srl_params, lr=1e-3, weight_decay=0.01)
epochs = 20
scheduler = get_linear_schedule_with_warmup(optimizer, int(epochs*len(srl_train_loader)*0.1), epochs*len(srl_train_loader))
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print(f"\nSRL v2: {epochs} epochs, BiLSTM-512, {len(srl_train_loader)} batches/epoch\n")

best_srl = 0.802
for epoch in range(epochs):
    model.train(); tl = 0.0
    for batch in tqdm(srl_train_loader, desc=f"SRL E{epoch+1}"):
        logits = fwd_srl_v2(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["pred_mask"].to(device))
        loss = loss_fn(logits.view(-1, len(srl_tag_list)), batch["labels"].to(device).view(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); tl += loss.item()
    model.eval(); all_g, all_p = [], []
    with torch.no_grad():
        for batch in srl_dev_loader:
            logits = fwd_srl_v2(batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["pred_mask"].to(device))
            logits_cpu = logits.cpu(); labs = batch["labels"]
            for j in range(labs.size(0)):
                vi = (labs[j] != -100).nonzero(as_tuple=True)[0]
                if len(vi) == 0: continue
                vl = logits_cpu[j, vi]
                vd = viterbi_decode(vl, srl_map)
                gl = [srl_tag_list[labs[j, k]] for k in vi.tolist()]
                pl = [srl_tag_list[vd[ki]] if vd[ki] < len(srl_tag_list) else "O" for ki in range(len(vi))]
                # Exclude V from scoring
                gl_filtered = [g if g != "V" else "O" for g in gl]
                pl_filtered = [p if p != "V" else "O" for p in pl]
                all_g.append(gl_filtered); all_p.append(pl_filtered)
    f1 = seq_f1(all_g, all_p)
    print(f"SRL E{epoch+1}/{epochs} — Loss: {tl/len(srl_train_loader):.4f}, F1: {f1:.4f}")
    if f1 > best_srl:
        best_srl = f1
        best_state = {
            "srl_scalar_mix": model.srl_scalar_mix2.state_dict(),
            "srl_lstm": model.srl_lstm2.state_dict(),
            "srl_proj": model.srl_proj2.state_dict(),
            "srl_head": model.srl_head2.state_dict(),
        }

print(f"\n{'='*50}")
print(f"SRL v2 best F1: {best_srl:.4f}")
print(f"SRL v1 baseline: 0.802")
print(f"Delta: {best_srl - 0.802:+.4f}")
print(f"{'='*50}")

if best_srl > 0.802:
    # Save to leader — rename keys back to canonical names
    for k, v in best_state.items():
        state[k] = v
    head_scores["srl"] = {"score": round(best_srl, 4), "metric": "F1"}
    save_leader(state, f"v3.3_srl{round(best_srl*1000)}", head_scores)
    print("Saved to leader!")
else:
    print("No improvement — keeping v1 SRL head.")

# Cleanup
del model.srl_scalar_mix2, model.srl_lstm2, model.srl_proj2, model.srl_head2
torch.cuda.empty_cache()

SRL data: 363,488 train, 1,236 dev
SRL v2 trainable: 8,443,972

SRL v2: 20 epochs, BiLSTM-512, 5680 batches/epoch



SRL E1: 100%|██████████| 5680/5680 [17:05<00:00,  5.54it/s]


SRL E1/20 — Loss: 0.4536, F1: 0.7636


SRL E2: 100%|██████████| 5680/5680 [17:03<00:00,  5.55it/s]


SRL E2/20 — Loss: 0.3121, F1: 0.7519


SRL E3:  37%|███▋      | 2102/5680 [06:19<10:45,  5.55it/s]


KeyboardInterrupt: 